![FLIP Banner](../../Assets/images/flip-banner.png)

# FLIP: Agentic AI in Practice
**Module 08: Advanced Agentic AI**

---

## Session 8D: MCP Fundamentals — Tool and Context Servers

<div align="center">

<table>
<thead><tr><th><strong>Item</strong></th><th><strong>Description</strong></th></tr></thead>
<tbody>
<tr><td align="left">Estimated time</td><td>2 hours</td></tr>
<tr><td align="left">Mandatory part</td><td>Local mock MCP-style client/server workflow</td></tr>
<tr><td align="left">Optional part</td><td>Real MCP SDK exploration if available</td></tr>
<tr><td align="left">Main output</td><td>A mock MCP server exposing tools, resources and prompts with capability checks</td></tr>
</tbody>
</table>

</div>

---

**Table of Contents**

1. [Overview and Learning Goals](#m08d-overview)
2. [Setup and Background](#m08d-setup)
3. [Core Concepts](#m08d-concepts)
4. [Guided Implementation](#m08d-implementation)
5. [Testing and Analysis](#m08d-testing)
6. [Student Tasks](#m08d-tasks)
7. [Submission and Reflection](#m08d-submission)

---

<a id="m08d-overview"></a>

### 1. Overview and Learning Goals

MCP, or Model Context Protocol, is an advanced integration concept for agentic AI. Earlier notebooks defined tools directly inside a Python notebook, which is excellent for learning but does not scale: every new agent would need its own copy of every tool. Real agent systems need a portable way to connect AI applications to tools and context, in the same way that USB gave computers one standard plug for many devices. MCP plays that role: one protocol, many servers, many capabilities.

A simplified MCP-style architecture looks like this:

```text
+--------------------+       +--------------+       +----------------------+
|  AI application    |       |  MCP client  |       |  MCP server          |
|  (host: an agent,  | <---> |  (speaks the | <---> |  (exposes            |
|  IDE or chat app)  |       |   protocol)  |       |   capabilities)      |
+--------------------+       +--------------+       +----------+-----------+
                                                               |
                                             +-----------------+----------------+
                                             |                 |                |
                                         +-------+       +-----------+     +---------+
                                         | Tools |       | Resources |     | Prompts |
                                         +-------+       +-----------+     +---------+
```

The host application talks to an MCP client; the client talks to one or more MCP servers; each server exposes three kinds of capability: tools (actions), resources (readable context) and prompts (reusable templates).

This notebook does not require a real MCP server. It uses a local mock server so you can understand the structure without any installation or network access: capability discovery, resources, prompts, tool calls, argument validation and safe rejection of unsupported capabilities. By the end of the session you should be able to explain the client/server split, read and extend a server manifest, trace a tool call from discovery to validated execution, and justify why the server — not the client — must enforce the final safety checks.

<a id="m08d-setup"></a>

### 2. Setup and Background

The mandatory part of this session uses only the Python standard library, so it runs unchanged in Google Colab or local Jupyter with no accounts, keys or installations. That is deliberate: the lesson here is protocol structure, not vendor tooling, and a mock server you can read end-to-end teaches structure better than a black box.

The safety boundary for this session is the same one used across Module 08: only synthetic public-style teaching data, no private documents or credentials, no real external side effects, and every response must be inspectable. If you later experiment with a real MCP SDK in the optional section, never hard-code a token into a cell — load secrets from environment variables or prompt for them with `getpass` — and never connect to an MCP server you do not trust, because a malicious server can advertise innocent-looking capabilities that do harmful things when invoked.

Run the setup cell below first. If a later cell raises `NameError`, it almost always means the cells were run out of order; restart the kernel and run all cells from the top.

In [ ]:
# Standard-library imports only: the mock MCP workflow is fully offline by
# design, so the protocol structure can be studied without any installation.
import json   # pretty-printing manifests and responses for inspection
import re     # tokenisation for the small public-notes search tool
from typing import Any, Dict, List   # type hints document each interface

print("M08D MCP setup complete.")

<a id="m08d-concepts"></a>

### 3. Core Concepts

MCP-style systems separate **capability discovery** from **capability invocation**. Discovery tells a client what the server claims to provide. Invocation is the actual tool call or resource read. These are different stages and both require controls.

A restaurant analogy makes the split concrete: the manifest is the menu — it tells you what you may order — but the kitchen still checks every order before cooking. A dish being printed on the menu does not mean any order for it is acceptable, and a dish not on the menu must be politely declined rather than improvised.

A typical exchange between client and server looks like this:

```text
Client                                        Server
  |                                             |
  |  1. discover()  ------------------------->  |
  |  <-- manifest: tools, resources, prompts    |
  |                                             |
  |  2. call_tool("rectangle_area",             |
  |        {"width": 3, "height": 4}) ------->  |  check tool exists
  |                                             |  validate arguments
  |                                             |  execute tool
  |  <-- {"ok": true, "result": {"value": 12}}  |
  |                                             |
  |  3. call_tool("delete_all_files", {}) --->  |  unknown tool
  |  <-- {"ok": false, "error": "unknown..."}   |  (fails safely, no side effect)
```

Core concepts:

<div align="center">

<table>
<thead><tr><th><strong>Concept</strong></th><th><strong>Meaning</strong></th><th><strong>Risk</strong></th></tr></thead>
<tbody>
<tr><td align="left">Tool</td><td>A callable action such as search, calculate or create ticket.</td><td>May create side effects if poorly scoped.</td></tr>
<tr><td align="left">Resource</td><td>Readable context such as a document, schema or note.</td><td>May leak private data if exposed incorrectly.</td></tr>
<tr><td align="left">Prompt</td><td>A reusable instruction template.</td><td>May encode unsafe or over-broad behaviour.</td></tr>
<tr><td align="left">Manifest</td><td>A description of exposed capabilities.</td><td>Not a security guarantee by itself.</td></tr>
</tbody>
</table>

</div>

A standard interface does not automatically make an agent safe. The server still needs argument validation, capability scoping, logging and refusal for unknown or unsafe calls. Keep that last row in mind throughout: a manifest is a claim, not a proof.

<a id="m08d-implementation"></a>

### 4. Guided Implementation

You will now build the mock system in three layers: the manifest that describes capabilities, the server that validates and executes them, and the client that consumes them. Read each explanation before running the code, and predict the output first.

#### 4.1 MCP-Style Server Manifest

The manifest below is a local teaching version of an MCP server description. It lists two tools, two resources and one prompt, and gives each tool a declared `risk` level and a `required_args` list. Those two fields are design decisions worth noticing: declaring required arguments up front lets a client fail fast on malformed calls, and declaring risk makes review possible before anything is exposed to an agent. The client can discover these capabilities, but the server must still validate every request — the manifest describes, it does not enforce.

In [ ]:
# The manifest is plain data that DESCRIBES capabilities. Enforcement lives in
# the server code, never here: a description cannot defend itself.
MCP_SERVER_MANIFEST = {
    "server_name": "flip_teaching_mcp_server",
    "version": "0.1.0",
    "tools": {
        "rectangle_area": {
            "description": "Calculate rectangle area from non-negative width and height.",
            "required_args": ["width", "height"],   # declared so clients can fail fast
            "risk": "low"                            # declared so reviewers can scope exposure
        },
        "search_public_notes": {
            "description": "Search approved public teaching notes.",
            "required_args": ["query"],
            "risk": "low"
        },
    },
    "resources": {
        # Resource URIs use a scheme-like naming convention so that approved
        # context is addressable and auditable, exactly as in real MCP.
        "resource://unit/syllabus": {
            "description": "Synthetic public unit syllabus summary.",
            "risk": "low"
        },
        "resource://unit/safety-rules": {
            "description": "Synthetic public safety rules.",
            "risk": "low"
        },
    },
    "prompts": {
        "safe_context_answer": {
            "description": "Answer using only approved context.",
            "variables": ["question", "context"]
        }
    }
}

# The approved-content store is separate from the manifest on purpose:
# listing a resource and being able to serve it are two different checks.
APPROVED_RESOURCES = {
    "resource://unit/syllabus": "FLIP covers foundations, Flowise, LangChain, RAG, LangGraph, multi-agent systems, safety, model adaptation and advanced agents.",
    "resource://unit/safety-rules": "Do not use private data, hidden instructor materials, credentials, shell commands, email sending or unsafe external side effects."
}

PUBLIC_NOTES = [
    "MCP-style servers expose tools, resources and prompts to clients.",
    "A tool call should be validated before execution.",
    "Resources provide readable context but should not contain private data.",
    "Prompts can be reused as workflow templates.",
]

print(json.dumps(MCP_SERVER_MANIFEST, indent=2))

#### 4.2 Local MCP Client/Server Workflow

The mock server implements four behaviours, one per capability type plus discovery:

```text
1. list_capabilities()          -> return the manifest (discovery)
2. read_resource(uri)           -> serve approved context only
3. get_prompt(name)             -> return a reusable template
4. call_tool(tool_name, args)   -> validate, then execute
```

Notice where validation lives: the server, not the client, enforces the final checks. This is important because a client may be buggy, compromised or simply too permissive, and in a real deployment the server cannot know which. Trust boundaries belong on the side that owns the capability. The tool functions themselves are kept deliberately tiny and deterministic so that all the interesting behaviour — validation, refusal, structured errors — is easy to see.

Every response uses the same `{"ok": ..., "error": ..., "result": ...}` envelope you met earlier in the module, so a client can always check success the same way. Expect three kinds of outcome when you experiment below: a valid call succeeds with a result, a malformed call fails with a named validation error, and an unknown capability fails safely without any side effect.

In [ ]:
# Tool implementations: deliberately tiny and deterministic, so the teaching
# focus stays on the validation and refusal logic around them.

def rectangle_area(width: float, height: float) -> float:
    return width * height

def word_count(text: str) -> int:
    return len(text.split())

def validate_rectangle_args(args: Dict[str, Any]) -> Dict[str, Any]:
    # Validation is a separate function per tool: each tool owns its own
    # contract, and each rejection names the exact rule that was broken.
    if not isinstance(args, dict):
        return {"ok": False, "error": "arguments must be a dictionary", "result": None}
    for key in ["width", "height"]:
        if key not in args:
            return {"ok": False, "error": f"missing argument: {key}", "result": None}
        try:
            value = float(args[key])   # accept "3" as well as 3, but insist on numeric
        except (TypeError, ValueError):
            return {"ok": False, "error": f"{key} must be numeric", "result": None}
        if value < 0:
            return {"ok": False, "error": f"{key} must be non-negative", "result": None}
        args[key] = value               # store the coerced value for execution
    return {"ok": True, "error": None, "result": args}

def validate_word_count_args(args: Dict[str, Any]) -> Dict[str, Any]:
    if not isinstance(args, dict) or "text" not in args:
        return {"ok": False, "error": "missing text argument", "result": None}
    if not isinstance(args["text"], str) or not args["text"].strip():
        return {"ok": False, "error": "text must be a non-empty string", "result": None}
    return {"ok": True, "error": None, "result": {"text": args["text"]}}

def search_public_notes(query: str) -> List[str]:
    # Keyword overlap again stands in for real retrieval: same role as an
    # embedding search, but fully inspectable in a teaching setting.
    query_terms = set(re.findall(r"[a-zA-Z_]+", query.lower()))
    results = []
    for note in PUBLIC_NOTES:
        note_terms = set(re.findall(r"[a-zA-Z_]+", note.lower()))
        if query_terms.intersection(note_terms):
            results.append(note)
    return results

In [ ]:
class MockMCPServer:
    # The server owns the trust boundary. Every public method answers one
    # question first: "is this capability known and approved?" — and only
    # then does any work.

    def __init__(self, manifest):
        self.manifest = manifest

    def list_capabilities(self):
        # Discovery: hand back the manifest. Harmless by itself, but remember
        # that everything listed here becomes visible to any connected agent.
        return {"ok": True, "error": None, "result": self.manifest}

    def read_resource(self, uri: str):
        # Two separate checks by design: the URI must be listed in the
        # manifest AND present in the approved store. A resource that is
        # advertised but not approved is served to no one.
        if uri not in self.manifest["resources"]:
            return {"ok": False, "error": f"unknown resource: {uri}", "result": None}
        if uri not in APPROVED_RESOURCES:
            return {"ok": False, "error": f"resource unavailable: {uri}", "result": None}
        return {"ok": True, "error": None, "result": {"uri": uri, "content": APPROVED_RESOURCES[uri]}}

    def get_prompt(self, name: str):
        if name not in self.manifest["prompts"]:
            return {"ok": False, "error": f"unknown prompt: {name}", "result": None}
        # The template constrains the answer to approved context only —
        # prompts are policy, not just convenience.
        template = "Question: {question}\nContext: {context}\nAnswer using only approved context."
        return {"ok": True, "error": None, "result": {"name": name, "template": template}}

    def call_tool(self, tool_name: str, args: Dict[str, Any]):
        # Gate 1: the tool must exist in the manifest. Unknown tools fail
        # safely with a structured error and, crucially, no side effect.
        if tool_name not in self.manifest["tools"]:
            return {"ok": False, "error": f"unknown tool: {tool_name}", "result": None}

        # Gate 2: per-tool argument validation, then execution.
        if tool_name == "rectangle_area":
            validation = validate_rectangle_args(dict(args))   # copy: never mutate caller input
            if not validation["ok"]:
                return validation
            return {"ok": True, "error": None, "result": {"tool": tool_name, "value": rectangle_area(**validation["result"])}}

        if tool_name == "search_public_notes":
            query = args.get("query") if isinstance(args, dict) else None
            if not isinstance(query, str) or not query.strip():
                return {"ok": False, "error": "query must be a non-empty string", "result": None}
            return {"ok": True, "error": None, "result": {"tool": tool_name, "value": search_public_notes(query)}}

        if tool_name == "word_count":
            # word_count is implemented and validated but NOT listed in the
            # manifest, so the manifest gate above refuses it before this
            # branch is ever reached. The student tasks ask you to enable it
            # properly by adding a manifest entry.
            validation = validate_word_count_args(args)
            if not validation["ok"]:
                return validation
            return {"ok": True, "error": None, "result": {"tool": tool_name, "value": word_count(validation["result"]["text"])}}

        # A manifest entry with no implementation is also refused explicitly.
        return {"ok": False, "error": f"tool not implemented: {tool_name}", "result": None}

In [ ]:
class MockMCPClient:
    # The client is thin on purpose: it forwards requests and composes
    # workflows, but it never re-implements the server's safety checks.
    # Duplicate enforcement drifts; single enforcement stays honest.

    def __init__(self, server):
        self.server = server

    def discover(self):
        return self.server.list_capabilities()

    def call_tool(self, tool_name: str, args: Dict[str, Any]):
        return self.server.call_tool(tool_name, args)

    def answer_with_resource(self, question: str, resource_uri: str):
        # A two-step composed workflow: fetch approved context, then apply
        # the approved prompt template. Each step can fail independently,
        # and a failure at either step is returned unchanged to the caller.
        resource = self.server.read_resource(resource_uri)
        if not resource["ok"]:
            return resource
        prompt = self.server.get_prompt("safe_context_answer")
        if not prompt["ok"]:
            return prompt
        return {
            "ok": True,
            "error": None,
            "result": {
                "question": question,
                "resource_uri": resource_uri,
                "answer": "Based on the approved resource, " + resource["result"]["content"],
                "limitations": ["This answer uses only the selected approved resource."]
            }
        }

server = MockMCPServer(MCP_SERVER_MANIFEST)
client = MockMCPClient(server)

print(client.call_tool("rectangle_area", {"width": 3, "height": 4}))
print(client.answer_with_resource("What does the unit cover?", "resource://unit/syllabus"))

#### 4.3 Inspection and Capability Boundaries

Now probe the boundary deliberately. The cell below sends a mixture of valid calls, an unknown tool, and an invalid argument, then displays each response. As you read the output, check each response against this list:

```text
1. Was the capability known?             (unknown tools must fail, not improvise)
2. Was the resource approved?            (listed AND served are separate checks)
3. Were arguments valid?                 (each rejection names the broken rule)
4. Was the result limited to approved data?
5. Did unknown tools fail safely?        (structured error, zero side effects)
```

The most important line in the output is the refusal of `delete_all_files`: nothing was deleted, no exception crashed the client, and the error message states exactly why. That combination — refuse, survive, explain — is what "fails safely" means.

Also watch the `word_count` call: the function exists in the server code, yet the call is refused with `unknown tool`. The manifest gate runs first, so an implemented-but-unlisted capability is treated exactly like a capability that does not exist. Exposure is an explicit decision, never an accident of implementation — you will enable this tool deliberately in the student tasks.

In [ ]:
def display_response(response):
    # Presentation only: errors print as one line, successes pretty-print,
    # so a scan of the output separates refusals from results at a glance.
    if not response.get("ok"):
        print("ERROR:", response.get("error"))
    else:
        print(json.dumps(response["result"], indent=2))

# A deliberate mixture: a valid call, an unknown tool, an invalid argument,
# an implemented-but-unlisted tool (word_count, refused by the manifest
# gate), and a composed resource-plus-prompt workflow.
for response in [
    client.call_tool("rectangle_area", {"width": 5, "height": 2}),
    client.call_tool("delete_all_files", {}),
    client.call_tool("rectangle_area", {"width": -5, "height": 2}),
    client.call_tool("word_count", {"text": "MCP exposes tools resources prompts"}),
    client.answer_with_resource("What are the safety rules?", "resource://unit/safety-rules"),
]:
    print("\n---")
    display_response(response)

#### 4.4 Optional Real MCP SDK Section

This section is optional. A real MCP SDK can replace the mock client/server, but the safety principles are unchanged:

```text
1. Expose only necessary capabilities.
2. Validate every argument.
3. Avoid private resources unless authorised.
4. Log calls.
5. Reject unknown or high-risk capabilities.
```

Do not connect to an untrusted MCP server in this lab. If you experiment outside the lab, treat a third-party MCP server with the same suspicion as a third-party browser extension: it runs with whatever access you grant it, and its manifest is marketing until you have reviewed what the capabilities actually do. If the SDK is not available in your environment, record the skip note and move on — the mandatory learning is complete without it.

In [ ]:
# Optional real-SDK placeholder. Returning False by default is deliberate:
# the notebook must pass for every student, including those with no network
# access and no installed MCP SDK.

def optional_real_mcp_available():
    return False

if not optional_real_mcp_available():
    print("Skipped: real MCP SDK not configured in this environment.")

<a id="m08d-testing"></a>

### 5. Testing and Analysis

The tests below pin down the server's contract from both directions: what must succeed and what must be refused. Discovery must return a manifest containing tools; a valid tool call must return the correct value; a negative dimension must be rejected; an unknown tool such as `send_email` must fail without side effects; an approved resource must be served while an unlisted private URI is refused; and the implemented-but-unlisted `word_count` tool must be refused by the manifest gate, proving that exposure requires an explicit manifest entry. If any single assert fails, the error tells you which side of the boundary broke: a failing success-case means the capability path is damaged, and a failing refusal-case means the boundary has a hole — the second kind is the more dangerous, because nothing crashes when a hole opens.

In [ ]:
# Success side of the contract: discovery and a valid call must work.
manifest_response = client.discover()
assert manifest_response["ok"] is True
assert "tools" in manifest_response["result"]

area = client.call_tool("rectangle_area", {"width": 6, "height": 7})
assert area["ok"] is True
assert area["result"]["value"] == 42

# Refusal side: invalid arguments must be rejected, not silently corrected.
negative = client.call_tool("rectangle_area", {"width": -1, "height": 7})
assert negative["ok"] is False

# Refusal side: unknown capabilities must fail safely with no side effect.
unknown = client.call_tool("send_email", {"to": "someone@example.com"})
assert unknown["ok"] is False

# Resources: approved content is served, unlisted private URIs are refused.
resource = server.read_resource("resource://unit/safety-rules")
assert resource["ok"] is True
assert "private data" in resource["result"]["content"]

unknown_resource = server.read_resource("resource://private/grades")
assert unknown_resource["ok"] is False

# word_count is implemented in the server but NOT listed in the manifest,
# so the manifest gate must refuse it. Enabling it properly is Task 2.
wc = client.call_tool("word_count", {"text": "one two three"})
assert wc["ok"] is False
assert "unknown tool" in wc["error"]

print("All M08D mandatory MCP-style tests passed.")

<a id="m08d-tasks"></a>

### 6. Student Tasks

Complete the tasks below using only the local mock system. For each programming task, state what you expect in the normal case, at the edges, and on failure — then show that the code agrees with you.

<div align="center">

<table>
<thead>
<tr><th><strong>Task</strong></th><th><strong>What you need to do</strong></th><th><strong>Why it matters</strong></th><th><strong>Expected evidence</strong></th></tr>
</thead>
<tbody>
<tr><td align="left">Task 1: Run baseline tests</td><td>Run all mandatory cells from the top. Normal: every assert passes. Failure: a <code>NameError</code> means cells ran out of order — restart and run all.</td><td>Confirms the reference behaviour before you change anything.</td><td>Output showing <code>All M08D mandatory MCP-style tests passed.</code></td></tr>
<tr><td align="left">Task 2: Enable the hidden tool</td><td>Add <code>word_count</code> to the manifest with a description, <code>required_args</code> and a risk level, confirm the call now succeeds (value 3 for "one two three"), then add one further deterministic low-risk tool of your own design.</td><td>Exposure must be an explicit, reviewable decision; you practise keeping manifest and server in sync.</td><td>Updated manifest, server code, and a successful <code>word_count</code> call.</td></tr>
<tr><td align="left">Task 3: Validate arguments</td><td>Write a validator for your new tool that rejects missing arguments, wrong types and out-of-range values, each with a specific error message. Edge: decide explicitly how borderline inputs (empty string, zero, very large values) are treated.</td><td>Validation at the server is the trust boundary; vague error messages make debugging and auditing impossible.</td><td>Validation code plus examples of each rejection.</td></tr>
<tr><td align="left">Task 4: Add tests</td><td>Add assert-based tests covering one valid call (normal), one invalid-argument call (edge/failure) and one unknown-capability call (failure) for your tool, plus one test showing <code>word_count</code> now succeeds after Task 2.</td><td>Tests make the boundary checkable on every future change, not just today.</td><td>Passing test cell.</td></tr>
<tr><td align="left">Task 5: Analyse the security boundary</td><td>Explain in a short paragraph why tools such as shell execution or email sending should not be exposed by this server, and why a manifest alone cannot protect anything.</td><td>Articulating the boundary is how the design lesson transfers to real MCP deployments.</td><td>Short written paragraph.</td></tr>
</tbody>
</table>

</div>

<a id="m08d-submission"></a>

### 7. Submission and Reflection

Submit the completed notebook with:

```text
1. Baseline test output.
2. Updated manifest including word_count and your new tool.
3. Validation logic for your tool.
4. Added assert-based tests.
5. Security-boundary paragraph.
6. Optional real MCP output or skipped note.
7. 150–250 word reflection.
```

**Quality checks.** Restart the kernel and run all cells top to bottom before submitting: every assert must pass, your new tool must appear in both the manifest and the server, no cell may contain a real token or private URL, and each section heading and anchor must be intact.

**Debugging guide.** If `call_tool` returns `unknown tool` for your new tool, the manifest key and the name checked in `call_tool` do not match exactly. If a valid call returns a validation error, print the args dictionary just before validation — type coercion (string versus number) is the usual culprit. If the baseline tests fail after your edits, you have changed shared objects; check that you extended the manifest rather than replacing it. If `NameError` appears anywhere, restart and run all cells in order.

**Reflection questions.**

1. Why are discovery and invocation treated as separate stages, and what could go wrong if a client trusted discovery alone?
2. Why must the server, rather than the client, enforce the final validation?
3. What does the `word_count` gap teach about the difference between an implemented capability and a documented one?
4. How would you decide whether a new tool is safe enough to add to a server's manifest?
5. What additional controls would a real MCP deployment need that this mock omits (for example authentication, rate limits, logging)?

#### Further Readings

- MCP official introduction: <https://modelcontextprotocol.io/docs/getting-started/intro>
- MCP specification: <https://modelcontextprotocol.io/specification/2025-06-18>
- MCP Python SDK: <https://github.com/modelcontextprotocol/python-sdk>
- OpenAI Agents SDK MCP guide: <https://openai.github.io/openai-agents-python/mcp/>
- LangChain MCP adapters: <https://docs.langchain.com/oss/python/langchain/mcp>